# BillingTokenizer Tutorial (Train, Save, Load, Use)

This notebook demonstrates how to use `BillingTokenizer` from `src/models.py` with the demo data file `data/mimic_4_icd_demo.parquet`.

You will learn how to:
1. Load and prepare code tokens from the dataset
2. Train a tokenizer vocabulary
3. Save tokenizer files
4. Load tokenizer from disk
5. Use the loaded tokenizer for encoding and decoding

## 1. Imports and paths

In [1]:
import pathlib
import sys
import json

import pandas as pd

repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root / "src"))

from models import BillingTokenizer, BillingDataset

print("Repo root:", repo_root)

Repo root: c:\Users\user\Documents\daniyal\biling-codes


## 2. Load demo data

We will load `mimic_4_icd_demo.parquet` and use the `icd_code_mapped` column as tokenizer input.

In [2]:
data_path = repo_root / "data" / "mimic_4_icd_demo_with_gaps.parquet"
df = pd.read_parquet(data_path)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Unique mapped ICD codes:", df["icd_code_mapped"].nunique())

df[["subject_id", "hadm_id", "chartdate", "icd_code_mapped"]].head()

Shape: (2639, 14)
Columns: ['subject_id', 'hadm_id', 'seq_num', 'chartdate', 'icd_code', 'icd_version', 'icd_code_mapped', 'pos', 'age_month', 'age_day', 'segment', 'Lo7', 'expired', 'expired_7']
Unique mapped ICD codes: 763


,subject_id,hadm_id,chartdate,icd_code_mapped
0,14546051,20000069,2131-11-10,0KQM0ZZ
1,14546051,20000069,2131-11-10,10E0XZZ
2,13074106,20000102,2135-10-25,M2060
3,13074106,20000102,2135-10-25,10907ZC
4,14990224,20000147,2121-08-30,02100Z9


## 3. Prepare tokens for training

`BillingTokenizer.train_from_list` in `models.py` expects an iterable and currently inserts `str(item)` into the vocabulary.

To build a code-level vocabulary, pass a flat list of code strings from `icd_code_mapped`.

In [3]:
# Flat list of mapped ICD codes (as strings)
train_codes = df["icd_code_mapped"].dropna().astype(str).tolist()

print("Training items:", len(train_codes))
print("Unique training codes:", len(set(train_codes)))
print("Sample:", train_codes[:10])

Training items: 2639
Unique training codes: 763
Sample: ['0KQM0ZZ', '10E0XZZ', 'M2060', '10907ZC', '02100Z9', 'B211YZZ', '021209W', '06BQ4ZZ', '5A1221Z', '0UL70ZZ']


## 4. Train a new BillingTokenizer

In [4]:
tokenizer = BillingTokenizer()
print("Initial vocab size:", tokenizer.vocab_size)

tokenizer.train_from_list(train_codes)
print("Trained vocab size:", tokenizer.vocab_size)

# Peek at a few vocab entries
first_items = list(tokenizer.vocab.items())[:15]
first_items

Initial vocab size: 5
Vocabulary trained. Size: 768
Trained vocab size: 768


[('[PAD]', 0),
 ('[UNK]', 1),
 ('[CLS]', 2),
 ('[SEP]', 3),
 ('[MASK]', 4),
 ('0045', 5),
 ('0046', 6),
 ('0047', 7),
 ('00500ZZ', 8),
 ('00510ZZ', 9),
 ('005T0ZZ', 10),
 ('00903ZX', 11),
 ('009100Z', 12),
 ('009630Z', 13),
 ('009U3ZX', 14)]

## 5. Save tokenizer to disk

`BillingTokenizer` inherits Hugging Face tokenizer save/load patterns, so `save_pretrained` writes tokenizer assets to a directory.

In [5]:
save_dir = repo_root / "saved_billing_tokenizer_tutorial"
save_dir.mkdir(parents=True, exist_ok=True)

tokenizer.save_pretrained(str(save_dir))

print("Saved files:")
for p in sorted(save_dir.iterdir()):
    print(" -", p.name)

Saved files:
 - special_tokens_map.json
 - tokenizer_config.json
 - vocab.json


## 6. Load tokenizer from disk

In [6]:
loaded_tokenizer = BillingTokenizer.from_pretrained(str(save_dir))

print("Loaded vocab size:", loaded_tokenizer.vocab_size)
print("Loaded pad token id:", loaded_tokenizer.pad_token_id)
print("Loaded cls token id:", loaded_tokenizer.cls_token_id)

assert loaded_tokenizer.vocab_size == tokenizer.vocab_size

Loaded vocab size: 768
Loaded pad token id: 0
Loaded cls token id: 2


## 7. Use tokenizer: encode and decode a sequence

In [7]:
example_codes = train_codes[:12]
print("Example code sequence:", example_codes)
example_codes = " ".join(example_codes)
encoded = loaded_tokenizer(
    example_codes,
    add_special_tokens=True,
    truncation=True,
    max_length=32,
)

print("\nInput IDs:", encoded["input_ids"])
print("Attention mask:", encoded["attention_mask"])

decoded_tokens = [loaded_tokenizer._convert_id_to_token(i) for i in encoded["input_ids"]]
print("Decoded tokens:", decoded_tokens)

Example code sequence: ['0KQM0ZZ', '10E0XZZ', 'M2060', '10907ZC', '02100Z9', 'B211YZZ', '021209W', '06BQ4ZZ', '5A1221Z', '0UL70ZZ', 'Q628', 'M2060']

Input IDs: [338, 496, 705, 492, 32, 607, 36, 106, 569, 438, 716, 705]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Decoded tokens: ['0KQM0ZZ', '10E0XZZ', 'M2060', '10907ZC', '02100Z9', 'B211YZZ', '021209W', '06BQ4ZZ', '5A1221Z', '0UL70ZZ', 'Q628', 'M2060']


In [8]:
# Pick one admission
sample_hadm = int(df["hadm_id"].iloc[12])
one_adm_df = df[df["hadm_id"] == sample_hadm].sort_values(["chartdate", "seq_num"])

# Get the codes for this admission
hadm_codes = one_adm_df["icd_code_mapped"].dropna().astype(str).tolist()
print("Admission:", sample_hadm)
print("Codes:", hadm_codes)

# Encode
hadm_encoded = loaded_tokenizer(
    " ".join(hadm_codes),
    add_special_tokens=True,
    truncation=True,
    max_length=64,
)

print("\nInput IDs:", hadm_encoded["input_ids"])
print("Attention mask:", hadm_encoded["attention_mask"])

# Decode
hadm_decoded = [loaded_tokenizer._convert_id_to_token(i) for i in hadm_encoded["input_ids"]]
print("Decoded tokens:", hadm_decoded)

# Verify round-trip
print("\nOriginal codes:  ", hadm_codes)
print("Decoded (no special):", [t for t in hadm_decoded if not t.startswith("[")])

Admission: 20000235
Codes: ['5A1D70Z', 'GAP_1', 'GAP_1', 'GAP_1', 'GAP_1', '4A020N8', 'B2000ZZ', 'GAP_5', 'GAP_1', 'GAP_1', '0DJD8ZZ']

Input IDs: [576, 664, 664, 664, 664, 545, 603, 667, 664, 664, 242]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Decoded tokens: ['5A1D70Z', 'GAP_1', 'GAP_1', 'GAP_1', 'GAP_1', '4A020N8', 'B2000ZZ', 'GAP_5', 'GAP_1', 'GAP_1', '0DJD8ZZ']

Original codes:   ['5A1D70Z', 'GAP_1', 'GAP_1', 'GAP_1', 'GAP_1', '4A020N8', 'B2000ZZ', 'GAP_5', 'GAP_1', 'GAP_1', '0DJD8ZZ']
Decoded (no special): ['5A1D70Z', 'GAP_1', 'GAP_1', 'GAP_1', 'GAP_1', '4A020N8', 'B2000ZZ', 'GAP_5', 'GAP_1', 'GAP_1', '0DJD8ZZ']


## 8. Use tokenizer on grouped admissions (`tokenize_df`) 

This demonstrates model-ready tokenization with optional `segment_ids`, `age_ids`, and tensor outputs.

In [22]:
# Build one admission-level dataframe
sample_hadm = int(df["hadm_id"].iloc[12])
one_adm_df = df[df["hadm_id"] == sample_hadm].sort_values(["chartdate", "seq_num"])

tok_out = loaded_tokenizer.tokenize_df(
    one_adm_df,
    code_column="icd_code_mapped",
    padding="max_length",
    truncation=True,
    max_length=64,
)

print("Admission:", sample_hadm)
print("Returned keys:", list(tok_out.keys()))
print("input_ids length:", len(tok_out["input_ids"]))
print("attention_mask length:", len(tok_out["attention_mask"]))
print("segment_ids length:", len(tok_out.get("segment_ids", [])))
print("age_ids length:", len(tok_out.get("age_ids", [])))

Admission: 20000235
Returned keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids length: 64
attention_mask length: 64
segment_ids length: 0
age_ids length: 0


## 9. Use BillingDataset for model training

`BillingDataset` is a PyTorch `Dataset` wrapper for admission-level data.

What it does:
1. Accepts grouped admissions (`df.groupby('hadm_id')`) or a list of dictionaries
2. Calls `tokenizer.tokenize_dict(...)` inside `__getitem__`
3. Returns a dictionary of model-ready tensors (for example `input_ids`, `attention_mask`, `segment_ids`, `age_ids`)
4. Removes the singleton batch dimension so `DataLoader` can batch samples correctly

In [39]:
df.hadm_id.nunique()

500

In [43]:
# Group by admission so each sample corresponds to one hospital admission
adm_groups = df.sort_values(["hadm_id", "chartdate", "seq_num"]).groupby("hadm_id")
loaded_tokenizer.model_max_length = 64
# Build dataset (tokenization happens when you index/get items)
dataset = BillingDataset(
    adm_groups,
    loaded_tokenizer,
)

print("Number of admissions in dataset:", len(dataset))

# Inspect one training sample
sample = dataset[0]
print("Returned keys:", list(sample.keys()))
for k, v in sample.items():
    shape = tuple(v.shape) if hasattr(v, "shape") else type(v)
    print(f"{k}: {shape}")

Converting grouped data to list of dictionaries...
Number of admissions in dataset: 500
Returned keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids: (64,)
token_type_ids: (66,)
attention_mask: (64,)


In [44]:
# Optional: use with DataLoader for batching
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=4, shuffle=False)
batch = next(iter(loader))

print("Batch keys:", list(batch.keys()))
for k, v in batch.items():
    shape = tuple(v.shape) if hasattr(v, "shape") else type(v)
    print(f"{k}: {shape}")

Batch keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids: (4, 64)
token_type_ids: (4, 66)
attention_mask: (4, 64)


## 10. (Optional) Save encoded sample to JSON for inspection

In [23]:
artifact_path = repo_root / "data" / "billing_tokenizer_sample_encoding.json"
serializable = {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in tok_out.items()}

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)

print("Wrote:", artifact_path)

Wrote: c:\Users\user\Documents\daniyal\biling-codes\data\billing_tokenizer_sample_encoding.json


You now have a complete workflow:
- train tokenizer from dataset codes
- save tokenizer with `save_pretrained`
- load tokenizer with `from_pretrained`
- encode real admissions with `tokenize_df`
- wrap admission data in `BillingDataset` for PyTorch training